# Chapter 4: Implementing a GPT Model from Scratch

Reference: `LLMs-from-scratch/ch04/`

**Goals:**
- Build the full GPT architecture (transformer blocks, layer norm, GELU)
- Implement the feed-forward network
- Assemble GPT-2 124M model
- Test generation with random weights

In [19]:
import sys
sys.path.append('..')
import torch.nn as nn
import torch
from src.config import GPT_CONFIG_124M
import tiktoken

## 1. Transformer Block

In [7]:
import sys; sys.path.append('..')
from src.config import GPT_CONFIG_124M
from src.attention import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg["emb_dim"])
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_resid = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_resid(x)
        x = x + shortcut  # residual connection

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_resid(x)
        x = x + shortcut  # residual connection
        return x

# Test
from src.config import GPT_CONFIG_124M
block = TransformerBlock(GPT_CONFIG_124M)
x = torch.randn(2, 4, 768)
out = block(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

Input shape:  torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


## 2. Layer Normalization

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.gamma = nn.Parameter(torch.ones(emb_dim)) # Learnable scale parameter
        self.beta = nn.Parameter(torch.zeros(emb_dim)) # Learnable shift parameter

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * norm + self.beta

#Test
ln = LayerNorm(emb_dim=768)
sample_input = torch.randn(2, 10, 768) # batch_size=2, seq_len=10, emb_dim=768
output = ln(sample_input)
print(f"Input mean:  {sample_input.mean().item():.4f}, var:  {sample_input.var().item():.4f}")
print(f"Output mean: {output.mean().item():.4f}, var: {output.var().item():.4f}")

Input mean:  0.0033, var:  1.0075
Output mean: 0.0000, var: 1.0001


## 3. Feed-Forward Network (GELU)

In [6]:
class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(0.79788456 * (x + 0.044715 * x ** 3))) # GELU activation function
    
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim), # Expand to 4x emb_dim
            GELU(), # Activation function
            nn.Linear(4 * emb_dim, emb_dim) # Project back to emb_dim
        )
    def forward(self, x):
        return self.layers(x)
    
# Test
ff = FeedForward(emb_dim=768)
x = torch.randn(2,4,768)
out = ff(x)
print(f"input Shape: {x.shape}")
print(f"Output Shape: {out.shape}")

input Shape: torch.Size([2, 4, 768])
Output Shape: torch.Size([2, 4, 768])


## 4. Full GPT Model Assembly

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        b, n = in_idx.shape
        token_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(n, device=in_idx.device))
        x = token_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

# Test
model = GPTModel(GPT_CONFIG_124M)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

x = torch.tensor([[0, 1, 2, 3], [4, 5, 6, 7]])
logits = model(x)
print(f"Input shape:  {x.shape}")
print(f"Logits shape: {logits.shape}")

Total parameters: 163,009,536
Input shape:  torch.Size([2, 4])
Logits shape: torch.Size([2, 4, 50257])


## 5. Text Generation (untrained)

In [21]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

# Test generation
start_context = "Hello, I am"
tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # add batch dim

print(f"Starting with: '{start_context}'")
print(f"Encoded shape: {encoded_tensor.shape}")

model.eval()  # set to eval mode (disables dropout)
out = generate_text_simple(model, encoded_tensor, max_new_tokens=6, context_size=GPT_CONFIG_124M["context_length"])
decoded = tokenizer.decode(out[0].tolist())

print(f"Generated:    '{decoded}'")

Starting with: 'Hello, I am'
Encoded shape: torch.Size([1, 4])
Generated:    'Hello, I am cavalry analogous commanding Marinasterdamado'
